In [ ]:
import pandas as pd
from tqdm import tqdm
import os
import librosa

import pandas as pd
import numpy as np

In [ ]:
def features_extractor(file):
    audio, sample_rate = librosa.load(file_name, res_type='kaiser_fast') 
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfccs_scaled_features = np.mean(mfccs_features.T,axis=0)
    return mfccs_scaled_features

In [ ]:
train = pd.read_csv('......./Downloads/metaData_CSS.csv')

In [ ]:
import numpy as np
from tqdm import tqdm

audio_dataset_path = '......./Downloads/dist/wav'

extracted_features=[]
for index_num,row in tqdm(train.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),str(row["filename"]))
    final_class_labels=row["label"]
    data=features_extractor(file_name)
    name = row["filename"]
    extracted_features.append([data,final_class_labels,name])

In [ ]:
extracted_features_df_train=pd.DataFrame(extracted_features,columns=['feature','label','name'])
extracted_features_df_train

In [ ]:
extracted_features_df_train['new_label'] = ''

In [ ]:
for i in range(len(extracted_features_df_train)):
    if extracted_features_df_train['label'][i] == 'positive':
        extracted_features_df_train['new_label'][i] = 1
    elif extracted_features_df_train['label'][i] == 'negative':
        extracted_features_df_train['new_label'][i] = 0

In [ ]:
features = extracted_features_df_train
features

In [ ]:
features.to_pickle('......./0810/20230309_893features.pkl')

In [ ]:
features = pd.read_pickle('......./0810/20230309_893features.pkl')

# cross_validation

## lr model

In [ ]:
features

In [ ]:
import pandas as pd
list1 = []
for i in range(0,40):
    list1.append(str(i))
print(list1)
df_mfcc = pd.DataFrame(features['feature'].to_list(), columns= list1)
df_mfcc


In [ ]:
df_mfcc1 = np.array(df_mfcc)
labels1 = np.int64(features['new_label'])

In [ ]:
name_all = np.array(features['name'])
 

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold

model = LogisticRegression()

In [ ]:
from sklearn.metrics import accuracy_score

from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from sklearn.metrics import r2_score, roc_auc_score, roc_curve, classification_report
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import auc

import matplotlib.pylab as plt
from scipy import interp


kf = StratifiedKFold(n_splits=10, shuffle=True,random_state=0)

acc_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
cnf_arr= []
x = 0

tprs_lr1 = []
aucs_lr1 = []
mean_fpr_lr1 = np.linspace(0, 1, 100)

fig, ax = plt.subplots(figsize=(6, 6))
    
for train_index, test_index in kf.split(df_mfcc1, labels1):
    X_train, X_test = df_mfcc1[train_index], df_mfcc1[test_index]
    y_train, y_test = labels1[train_index], labels1[test_index]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print('Accuracy Score: {:.4f}'.format(accuracy_score(y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(y_test, y_pred)))
    print("\n", classification_report(y_test, y_pred))
    
    cnf_matrix = confusion_matrix(y_test, y_pred)
    acc_arr[x] = accuracy_score(y_test, y_pred)
    f1_arr[x] = f1_score(y_test, y_pred)

    viz = RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=f"ROC fold {x}",
        alpha=0.3,
        lw=1,
        ax=ax,
    )
    
    interp_tpr_lr1 = np.interp(mean_fpr_lr1, viz.fpr, viz.tpr)
    interp_tpr_lr1[0] = 0.0
    tprs_lr1.append(interp_tpr_lr1)
    aucs_lr1.append(viz.roc_auc)
    
    x = x+ 1
    
ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")

print("%0.4f f1 score with a standard deviation of %0.4f" %
      (f1_arr.mean(), f1_arr.std()))
print("%0.4f accuracy with a standard deviation of %0.4f" %
      (acc_arr.mean(), acc_arr.std()))


mean_tpr_lr1 = np.mean(tprs_lr1, axis=0)
mean_tpr_lr1[-1] = 1.0
mean_auc_lr1 = auc(mean_fpr_lr1, mean_tpr_lr1)
std_auc_lr1 = np.std(aucs_lr1)
ax.plot(
    mean_fpr_lr1,
    mean_tpr_lr1,
    color="b",
    label=r"Mean ROC (AUC = %0.2f $\pm$ %0.2f)" % (mean_auc_lr1, std_auc_lr1),
    lw=2,
    alpha=0.8,
)

std_tpr_lr1 = np.std(tprs_lr1, axis=0)
tprs_upper = np.minimum(mean_tpr_lr1 + std_tpr_lr1, 1)
tprs_lower = np.maximum(mean_tpr_lr1 - std_tpr_lr1, 0)
ax.fill_between(
    mean_fpr_lr1,
    tprs_lower,
    tprs_upper,
    color="grey",
    alpha=0.2,
    label=r"$\pm$ 1 std. dev.",
)

ax.set(
    xlim=[-0.05, 1.05],
    ylim=[-0.05, 1.05],
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=f"Mean ROC curve with variability')",
)
ax.axis("square")
ax.legend(loc="lower right")
plt.show()


In [ ]:
plt.plot(mean_fpr_lr1, mean_tpr_lr1, label='LR (area = %0.2f)' % mean_auc_lr1, color = '#069AF3')
plt.legend(loc=4, fontsize=10)
plt.title(' ',fontsize=20)
plt.xlabel('False Positive Rate',fontsize=14)
plt.ylabel('True Positive Rate',fontsize=14)
plt.title("AUC(ROC curve)")
plt.plot([0, 1], [0, 1], color="green", lw=2, linestyle="--")

In [ ]:
np.save('......./Desktop/modeloo/20230216_4models/893_auc/mean_tpr_lr1',mean_tpr_lr1)

In [ ]:
np.load('......./Desktop/modeloo/20230216_4models/893_auc/mean_tpr_lr1.npy')

## svm model

In [ ]:

from sklearn.svm import SVC

model_svm = SVC(kernel='rbf',random_state=1,C=1,gamma=0.001,probability=True)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import auc
from sklearn.metrics import RocCurveDisplay

from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from sklearn.metrics import r2_score, roc_auc_score, roc_curve, classification_report
from sklearn.model_selection import KFold, StratifiedKFold
import matplotlib.pylab as plt



kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
cnf_arr= []
x = 0

tprs_svm1 = []
aucs_svm1 = []
mean_fpr_svm1 = np.linspace(0, 1, 100)

fig, ax = plt.subplots(figsize=(6, 6))


for train_index, test_index in kf.split(df_mfcc1, labels1):
    X_train, X_test = df_mfcc1[train_index], df_mfcc1[test_index]
    y_train, y_test = labels1[train_index], labels1[test_index]
    model_svm.fit(X_train, y_train)
    y_pred = model_svm.predict(X_test)
    print('Accuracy Score: {:.4f}'.format(accuracy_score(y_test, y_pred)))
    print('SVC f1-score  : {:.4f}'.format(f1_score(y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(y_test, y_pred)))
    print("\n", classification_report(y_test, y_pred))
    
    cnf_matrix = confusion_matrix(y_test, y_pred)
    acc_arr[x] = accuracy_score(y_test, y_pred)
    f1_arr[x] = f1_score(y_test, y_pred)
    
    
    viz = RocCurveDisplay.from_estimator(
        model_svm,
        X_test,
        y_test,
        name=f"ROC fold {x}",
        alpha=0.3,
        lw=1,
        ax=ax,
    )
    
    interp_tpr_svm1 = np.interp(mean_fpr_svm1, viz.fpr, viz.tpr)
    interp_tpr_svm1[0] = 0.0
    tprs_svm1.append(interp_tpr_svm1)
    aucs_svm1.append(viz.roc_auc)
    
    x = x+ 1
    
ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")


print("%0.4f f1 score with a standard deviation of %0.4f" %
      (f1_arr.mean(), f1_arr.std()))
print("%0.4f accuracy with a standard deviation of %0.4f" %
      (acc_arr.mean(), acc_arr.std()))


mean_tpr_svm1 = np.mean(tprs_svm1, axis=0)
mean_tpr_svm1[-1] = 1.0
mean_auc_svm1 = auc(mean_fpr_svm1, mean_tpr_svm1)
std_auc_svm1 = np.std(aucs_svm1)
ax.plot(
    mean_fpr_svm1,
    mean_tpr_svm1,
    color="b",
    label=r"Mean ROC (AUC = %0.2f $\pm$ %0.2f)" % (mean_auc_svm1, std_auc_svm1),
    lw=2,
    alpha=0.8,
)

std_tpr_svm1 = np.std(tprs_svm1, axis=0)
tprs_upper = np.minimum(mean_tpr_svm1 + std_tpr_svm1, 1)
tprs_lower = np.maximum(mean_tpr_svm1 - std_tpr_svm1, 0)
ax.fill_between(
    mean_fpr_svm1,
    tprs_lower,
    tprs_upper,
    color="grey",
    alpha=0.2,
    label=r"$\pm$ 1 std. dev.",
)

ax.set(
    xlim=[-0.05, 1.05],
    ylim=[-0.05, 1.05],
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=f"Mean ROC curve with variability')",
)
ax.axis("square")
ax.legend(loc="lower right")
plt.show()

In [ ]:
plt.plot(mean_fpr_svm1, mean_tpr_svm1, label='SVM(area = %0.2f)' % mean_auc_svm1, color = '#069AF3')
plt.legend(loc=4, fontsize=10)
plt.title(' ',fontsize=20)
plt.xlabel('False Positive Rate',fontsize=14)
plt.ylabel('True Positive Rate',fontsize=14)
plt.title("AUC(ROC curve)")
plt.plot([0, 1], [0, 1], color="green", lw=2, linestyle="--")

In [ ]:
np.save('......./Desktop/modeloo/20230216_4models/893_auc/mean_tpr_svm1',mean_tpr_svm1)

## cnn model

In [ ]:

import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import numpy as np
import librosa.display
import librosa

kf = StratifiedKFold(n_splits=10, shuffle=True,random_state=0)


a = 0
i = 0
i1 = 0


for train_index, test_index in kf.split(name_all,labels1):
    name_train, name_test = name_all[train_index], name_all[test_index]
    y_train, y_test = labels1[train_index], labels1[test_index]
    train_df = pd.DataFrame(name_train, columns = ['name'])
    train_df['label'] = ''
    train_df['label'] = y_train
    test_df = pd.DataFrame(name_test, columns = ['name'])
    test_df['label'] = ''
    test_df['label'] = y_test
    
    
    
    for f in train_df['name']:
        
        AUDIO_FILE = '......./Downloads/dist/wav/'
        samples, sample_rate = librosa.load(AUDIO_FILE + str(f), sr=None)
        sgram = librosa.stft(samples)
        sgram_mag, _ = librosa.magphase(sgram)
        mel_scale_sgram = librosa.feature.melspectrogram(S=sgram_mag, sr=sample_rate)
        mel_sgram = librosa.amplitude_to_db(mel_scale_sgram, ref=np.min)
        w = train_df.loc[train_df['name'] == f, 'label']
        if w[i] == 1:
            fig = plt.Figure()
            canvas = FigureCanvas(fig)
            ax = fig.add_subplot(111)
            img = librosa.display.specshow(mel_sgram, sr=sample_rate, ax = ax)
            images_dir = '......./Desktop/modeloo/20230216_4models/893_cnn/{0}/train/pos/'.format(a)
            fig.savefig(images_dir+ str(i)+ "_"+"pos" + "_"+ f.split(".")[0])
            i = i+1
        elif w[i] == 0:
            fig = plt.Figure()
            canvas = FigureCanvas(fig)
            ax = fig.add_subplot(111)
            img = librosa.display.specshow(mel_sgram, sr=sample_rate, ax = ax)
            images_dir = '......./Desktop/modeloo/20230216_4models/893_cnn/{0}/train/neg/'.format(a)
            fig.savefig(images_dir+ str(i)+ "_"+"neg" + "_"+ f.split(".")[0])
            i = i+1
        
    for f in test_df['name']:
        
        AUDIO_FILE = '......./Downloads/dist/wav/'
        samples, sample_rate = librosa.load(AUDIO_FILE + str(f), sr=None)
        sgram = librosa.stft(samples)
        sgram_mag, _ = librosa.magphase(sgram)
        mel_scale_sgram = librosa.feature.melspectrogram(S=sgram_mag, sr=sample_rate)
        mel_sgram = librosa.amplitude_to_db(mel_scale_sgram, ref=np.min)
        w = test_df.loc[test_df['name'] == f, 'label']
        if w[i1] == 1:
            fig = plt.Figure()
            canvas = FigureCanvas(fig)
            ax = fig.add_subplot(111)
            img = librosa.display.specshow(mel_sgram, sr=sample_rate, ax = ax)
            images_dir = '......./Desktop/modeloo/20230216_4models/893_cnn/{0}/test/pos/'.format(a)
            fig.savefig(images_dir+ str(i1)+ "_"+"pos" + "_"+ f.split(".")[0])
            i1 = i1+1
        elif w[i1] == 0:
            fig = plt.Figure()
            canvas = FigureCanvas(fig)
            ax = fig.add_subplot(111)
            img = librosa.display.specshow(mel_sgram, sr=sample_rate, ax = ax)
            images_dir = '......./Desktop/modeloo/20230216_4models/893_cnn/{0}/test/neg/'.format(a)
            fig.savefig(images_dir+ str(i1)+ "_"+"neg" + "_"+ f.split(".")[0])
            i1 = i1+1
            
    a = a+1
    i = 0
    i1 = 0
    

In [ ]:
train_index 

In [ ]:
path_train= '......./Desktop/modeloo/20230216_4models/893_cnn/0/train'
path_test= '......./Desktop/modeloo/20230216_4models/893_cnn/0/test'

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

datagen2 = ImageDataGenerator(rescale=1./255)
datagen = ImageDataGenerator(rescale=1. / 255, 
                                   rotation_range=40,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode='nearest')
train_generator = datagen2.flow_from_directory(
    path_train,
    target_size=(150,150),
    shuffle=True,
)


test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = datagen2.flow_from_directory(
    path_test,
    target_size=(150,150),
    shuffle=False,
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization, Input
import tensorflow as tf


modelIMG = tf.keras.models.Sequential([

    tf.keras.layers.Conv2D(8, (3,3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Conv2D(16, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(2, activation='softmax')
]) 

In [ ]:
modelIMG.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
history = modelIMG.fit(train_generator,batch_size=32,epochs=100)

In [ ]:
from tensorflow.keras.models import Sequential, save_model, load_model
# Load the model
filepath = '......./Desktop/modeloo/20230216_4models/893_cnn_model/0'
loaded_model_cnn1 = load_model(
    filepath,
    custom_objects=None,
    compile=True
)

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
import seaborn as sns
import  numpy as np
import matplotlib.pyplot as plt

accuracy = loaded_model_cnn1.evaluate(test_generator)
print('n', 'Test_Accuracy:-', accuracy[1])
pred = loaded_model_cnn1.predict(test_generator)
y_pred = np.argmax(pred, axis=1)
print('confusion matrix')
print(confusion_matrix(test_generator.classes, y_pred))

f, ax = plt.subplots(figsize=(8,5))
sns.heatmap(confusion_matrix(test_generator.classes, y_pred), annot=True, fmt=".0f", ax=ax)
plt.xlabel("y_pred")
plt.ylabel("y_true")
plt.show()
target_names = ['neg', 'pos']
print(classification_report(test_generator.classes, y_pred, target_names=target_names))

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import RocCurveDisplay
import numpy as np
from sklearn import metrics
from sklearn.metrics import auc
import matplotlib.pyplot as plt



FPR_cnn1, TPR_cnn1, thresholds_cnn1 = metrics.roc_curve(test_generator.classes,loaded_model_cnn1.predict(test_generator)[:,1])
AUC_cnn1 = auc(FPR_cnn1, TPR_cnn1)






plt.plot(FPR_cnn1,TPR_cnn1,label="CNN (area = %0.2f)" %AUC_cnn1,color='#069AF3',linestyle='-')


plt.legend(loc=4, fontsize=10)
plt.title(' ',fontsize=20)
plt.xlabel('False Positive Rate',fontsize=14)
plt.ylabel('True Positive Rate',fontsize=14)
plt.title("AUC(ROC curve)")



plt.plot([0, 1], [0, 1], color="green", lw=2, linestyle="--")
plt.show()

In [ ]:
tprs = []
aucs = []
mean_fpr = np.linspace(0,1,100)

tprs.append(interp(mean_fpr, FPR_cnn1, TPR_cnn1))
roc_auc = auc(FPR_cnn1, TPR_cnn1)
aucs.append(roc_auc)

In [ ]:
from tensorflow.keras.models import Sequential, save_model, load_model
filepath = '......./Desktop/modeloo/20230216_4models/893_cnn_model/0'
save_model(modelIMG, filepath)

In [ ]:
FPR_cnn1

In [ ]:
mean_tpr = np.mean(tprs, axis=0)

In [ ]:
mean_auc = auc(mean_fpr, mean_tpr)

In [ ]:
mean_tpr[-1] = 1.0

In [ ]:
mean_tpr[0]= 0

In [ ]:
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')
plt.plot(mean_fpr, mean_tpr, color='blue',
         label=r'Mean ROC (AUC = %0.2f )' % (mean_auc),lw=2, alpha=1)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC')
plt.legend(loc="lower right")
plt.text(0.32,0.7,'More accurate area',fontsize = 12)
plt.text(0.63,0.4,'Less accurate area',fontsize = 12)
plt.show()

In [ ]:
np.save('......./Desktop/modeloo/20230216_4models/893_auc/mean_tpr_cnn',mean_tpr)

## lstm

In [ ]:
features1 = features
features1

In [ ]:
features1.to_pickle('20230216features1.pkl')

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np


X_lstm =np.array(features1['feature'].tolist())
y_lstm =np.array(features1['new_label'].tolist())
X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_lstm, test_size=0.20, random_state=0)


In [ ]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from sklearn.model_selection import KFold
import numpy as np

from keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from keras.layers import *

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pylab as plt


from scipy import interp

from sklearn.metrics import r2_score, roc_auc_score, roc_curve, classification_report




# Define per-fold score containers
acc_per_fold = []
auc_per_fold = []
loss_per_fold = []
modelsss = []
 
kfold = StratifiedKFold(n_splits=10, shuffle=True,random_state=0)

# K-fold Cross Validation model evaluation
fold_no = 1

modelss = {}
tprs = []
aucs = []
mean_fpr = np.linspace(0,1,100)

for train, test in kfold.split(inputs, targets):

    print ('Creating model...')
    model = Sequential()

    model.add(Bidirectional(LSTM(128, return_sequences=True)))
    model.add(Dropout(0.3))
    model.add((Dense(128, activation='selu')))
    model.add(Dropout(0.3))
    model.add((Dense(64, activation='selu')))
    model.add(Dropout(0.3))
    model.add(Dense(1,activation='sigmoid'))
    #model.add(Dense(1))

    #print(model.summary())
    print ('Compiling...')
    model.compile(loss='mae',
                  optimizer='adam',
                  metrics=['accuracy'])
      # Generate a print
    print('------------------------------------------------------------------------')
    print(f'Training for fold {fold_no} ...')
                  
    num_epochs = 100
    num_batch_size = 32
    # Fit data to model
    history = model.fit(inputs[train], targets[train],
              batch_size=num_batch_size,
              epochs=num_epochs,
              verbose=1)
    # Generate generalization metrics
    scores = model.evaluate(inputs[test], targets[test], verbose=0)

    #print("\n", classification_report(y_pred, y_test))
    pred = model.predict(inputs[test])
    y_pred = [np.round(x) for x in pred]
    test_preds1 = [np.round(x) for x in y_pred]
    test_preds1 = np.array(test_preds1)
    test_preds1 = test_preds1.flatten('A')

    testtargets1 = np.array(targets[test])
    testtargets1 = testtargets1.flatten('A')

    print('confusion matrix')
    print(confusion_matrix(testtargets1, test_preds1))
    print("\n", classification_report(testtargets1, test_preds1))



    
    test_preds12 = np.array(pred)
    test_preds12 = test_preds12.flatten('A')


    fpr2, tpr2, thresholds2 = roc_curve(testtargets1, test_preds12)
    AUC_lstm21 = auc(fpr2, tpr2)
    print(f'auc: {AUC_lstm21}')

    
     

    tprs.append(interp(mean_fpr, fpr2, tpr2))
    roc_auc = auc(fpr2, tpr2)
    aucs.append(roc_auc)

    plt.plot(fpr2, tpr2, lw=2, alpha=0.3, label='ROC fold %d (AUC = %0.2f)' % (fold_no, AUC_lstm21))


    print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {scores[0]}; {model.metrics_names[1]} of {scores[1]*100}%')
    acc_per_fold.append(scores[1] * 100)
    auc_per_fold.append(AUC_lstm21)
    loss_per_fold.append(scores[0])
    modelsss.append(model)
    for index,acc in enumerate(acc_per_fold): 
        modelss[modelsss[index]] = acc
    print(modelss)

    # Increase fold number
    fold_no = fold_no + 1
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(acc_per_fold)):
    print('------------------------------------------------------------------------')
    print(f'> Fold {i+1} - Loss: {loss_per_fold[i]} - AUC: {auc_per_fold[i]} - Accuracy: {acc_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Accuracy: {np.mean(acc_per_fold)} (+- {np.std(acc_per_fold)})')
print(f'> Loss: {np.mean(loss_per_fold)}')
print(f'> AUC: {np.mean(auc_per_fold)} (+- {np.std(auc_per_fold)})')
print('------------------------------------------------------------------------')

plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')
mean_tpr = np.mean(tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)
plt.plot(mean_fpr, mean_tpr, color='blue',
         label=r'Mean ROC (AUC = %0.2f )' % (mean_auc),lw=2, alpha=1)
